In [8]:
import sqlite3
import pandas as pd

db_path = '../database/bank_churn.db'
con = sqlite3.connect(db_path)

def run_sql(query):
    """Performs SQL and returns DataFrame"""
    return pd.read_sql(query, con)

print("Connection is successful")

Connection is successful


In [9]:
run_sql("SELECT * FROM bank_customers LIMIT 5")

,rownumber,customerid,surname,first_name,date_of_birth,gender,marital_status,number_of_dependents,occupation,income,...,preferred_communication_channel,credit_score,credit_history_length,outstanding_loans,churn_flag,churn_reason,churn_date,balance,numofproducts,numcomplaints
0,1,83ef0b54-35f6-4f84-af58-5653ac0c0dc4,Smith,Troy,1987-08-29,Male,Divorced,3,Information systems manager,77710.14,...,Phone,397,24,41959.74,0,None,None,211359.05,1,0
1,2,009f115a-e5ca-4cf4-97d6-530140545e4e,Sullivan,Katrina,2000-02-07,Female,Married,1,Charity fundraiser,58209.87,...,Email,665,10,8916.67,0,None,None,30624.76,4,1
2,3,66309fd3-5009-44d3-a3f7-1657c869d573,Fuller,Henry,1954-02-03,Female,Single,1,Television production assistant,9794.01,...,Email,715,21,43270.54,0,None,None,111956.61,2,6
3,4,b02a30df-1a5f-4087-8075-2a35432da641,Young,Antonio,1991-01-15,Female,Divorced,5,Agricultural engineer,15088.98,...,Phone,747,17,17887.65,0,None,None,201187.61,1,0
4,5,0d932e5b-bb3a-4104-8c83-f84270f7f2ea,Andersen,John,1992-04-08,Female,Divorced,2,"Teacher, early years/pre",60726.56,...,Email,549,25,32686.84,0,None,None,60391.24,5,6


In [10]:
q="""
SELECT
    COUNT(*) as total_customers,
    SUM(churn_flag) as lost_customers,
    ROUND(CAST(SUM(churn_flag) AS FLOAT) / COUNT(*) * 100, 2) as churn_rate_percent
FROM bank_customers
"""
run_sql(q)

,total_customers,lost_customers,churn_rate_percent
0,115640,14094,12.19


In [11]:
q="""
SELECT
    education_level,
    COUNT(*) as total_customers,
    SUM(churn_flag) as lost_customers,
    ROUND(CAST(SUM(churn_flag) AS FLOAT) / COUNT(*) * 100, 2) as churn_rate
FROM bank_customers
GROUP BY education_level
ORDER BY churn_rate DESC
"""
run_sql(q)

,education_level,total_customers,lost_customers,churn_rate
0,Master's,28970,3572,12.33
1,Diploma,28950,3568,12.32
2,Bachelor's,28852,3497,12.12
3,High School,28868,3457,11.98


In [12]:
q_credit="""
SELECT 
    CASE
        WHEN credit_score < 300 THEN 'very poor'
        WHEN credit_score BETWEEN 300 AND 600 THEN 'poor'
        WHEN credit_score BETWEEN 600 AND 800 THEN 'fair'
        ELSE 'good'
    END as credit_group,
    COUNT(*) as total_people,
    ROUND(AVG(churn_flag) * 100, 2) as churn_rate
FROM bank_customers
GROUP BY 1
ORDER BY churn_rate DESC
"""

run_sql(q_credit)

,credit_group,total_people,churn_rate
0,poor,63329,16.84
1,fair,42011,7.22
2,good,10300,3.83


In [13]:
q_products = """
SELECT 
    numofproducts,
    COUNT(*) as total_people,
    ROUND(CAST(SUM(churn_flag) AS FLOAT) / COUNT(*) * 100, 2) as churn_rate

FROM bank_customers
GROUP BY numofproducts
ORDER BY churn_rate DESC
"""
run_sql(q_products)

,numofproducts,total_people,churn_rate
0,1,22898,20.87
1,2,23451,16.42
2,3,23198,11.40
3,4,23023,7.85
4,5,23070,4.40


In [14]:
df_full = run_sql('SELECT * FROM bank_customers')
numeric_cols = df_full.select_dtypes(include=['number']).columns
correlation = df_full[numeric_cols].corr()['churn_flag'].sort_values(ascending=False)

print(correlation)

churn_flag               1.000000
numcomplaints            0.204626
number_of_dependents     0.003109
credit_history_length    0.002899
income                   0.002286
customer_tenure          0.000344
outstanding_loans       -0.001146
rownumber               -0.001604
numofproducts           -0.179083
credit_score            -0.182802
balance                 -0.499981
Name: churn_flag, dtype: float64
